In [1]:
"""
Gemma 2 Logit Extraction for Crossword Clues

Requirements:
pip install transformers torch accelerate

You'll also need a Hugging Face token to access Gemma 2.
Get one at: https://huggingface.co/settings/tokens
"""

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import numpy as np

# Crossword clues
CLUES = {
    "1a": {"clue": "Acquisitive chap, as we see it", "length": 8},
    "5a": {"clue": "Back yard fencing weak and sagging", "length": 6},
    "8a": {"clue": "Stripping off uniform, love holding colonel's coat", "length": 10},
    "9a": {"clue": "Without a mark where they should be gained", "length": 4},
    "10a": {"clue": "Put a stop to Rugby's foul school leader", "length": 14},
    "11a": {"clue": "Foreign letter coming in is the French letter", "length": 7},
    "13a": {"clue": "Charge to pack knick-knacks hurriedly", "length": 7},
    "15a": {"clue": "At first, bear one fruit or another", "length": 7},
    "18a": {"clue": "Cited tot defending authoritarian leader", "length": 7},
    "21a": {"clue": "Heady mixture of qualities nurse developed", "length": 14},
    "22a": {"clue": "A little house might ... be this?", "length": 4},
    "23a": {"clue": "The writer's twice penning papers of merit", "length": 10},
    "24a": {"clue": "Highest-ranking lady has heart stolen roguishly", "length": 6},
    "25a": {"clue": "Grant a leader in the papers a column", "length": 8},
    "1d": {"clue": "Charles's going to loudly show amusement", "length": 7},
    "2d": {"clue": "Number one Scot with puritanical views", "length": 9},
    "3d": {"clue": "What handyman has flipping spoils apparel", "length": 7},
    "4d": {"clue": "Errant knight in huge trouble", "length": 7},
    "5d": {"clue": "Turned tenacious, holding on", "length": 9},
    "6d": {"clue": "Stuff's possibly tax-free, before cuts", "length": 7},
    "7d": {"clue": "Praise, primarily, to give the thing acclaim?", "length": 7},
    "12d": {"clue": "One admitted to being unstable? It's still to be settled", "length": 9},
    "14d": {"clue": "This vicar could be the record-holder", "length": 9},
    "16d": {"clue": "Complaint received by a miserly sort presenting a threat to workers", "length": 7},
    "17d": {"clue": "Liberal leader gave up splitting wood", "length": 7},
    "18d": {"clue": "Light vessel reversed by a kingdom once", "length": 7},
    "19d": {"clue": "Ruling line in colour, almost ugly", "length": 7},
    "20d": {"clue": "European bank invests — diving in here?", "length": 7}
}

def load_gemma2(model_name="google/gemma-2-9b-it", token=None):
    """Load Gemma 2 model and tokenizer"""
    print(f"Loading {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=token)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        token=token,
        device_map="auto",
        torch_dtype=torch.bfloat16
    )
    print("Model loaded successfully!")
    return model, tokenizer

def get_candidate_words(model, tokenizer, clue_text, word_length, num_candidates=10):
    """Get candidate words from the model for a cryptic crossword clue"""
    prompt = f"""This is a cryptic crossword clue. Suggest {num_candidates} possible answers.

Clue: {clue_text}
Length: {word_length} letters

Provide only the words, one per line, in uppercase:"""
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=0.7,
            do_sample=True,
            top_p=0.9
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response[len(prompt):].strip()
    
    # Extract candidate words
    candidates = []
    for line in response.split('\n'):
        word = line.strip().upper()
        # Remove numbering, bullets, etc.
        word = ''.join(c for c in word if c.isalpha())
        if len(word) == word_length:
            candidates.append(word)
    
    return candidates[:num_candidates]

def get_word_logits(model, tokenizer, word):
    """Get logit vector for a specific word"""
    inputs = tokenizer(word, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        # Get logits for the last token
        logits = outputs.logits[0, -1, :].cpu().numpy()
    
    return logits

def analyze_clue(model, tokenizer, clue_num, clue_data):
    """Analyze a single clue and return candidates with their logits"""
    print(f"\n{'='*80}")
    print(f"CLUE {clue_num}: {clue_data['clue']} ({clue_data['length']})")
    print(f"{'='*80}")
    
    # Get candidate words
    candidates = get_candidate_words(
        model, tokenizer, 
        clue_data['clue'], 
        clue_data['length']
    )
    
    if not candidates:
        print("No candidates found!")
        return {}
    
    print(f"\nFound {len(candidates)} candidates:")
    
    results = {}
    for candidate in candidates:
        print(f"\nCandidate: {candidate}")
        logits = get_word_logits(model, tokenizer, candidate)
        results[candidate] = logits
        
        # Show some statistics
        print(f"  Logit vector shape: {logits.shape}")
        print(f"  Mean logit: {np.mean(logits):.4f}")
        print(f"  Max logit: {np.max(logits):.4f}")
        print(f"  Min logit: {np.min(logits):.4f}")
        print(f"  Top 5 token indices: {np.argsort(logits)[-5:][::-1].tolist()}")
    
    return results

def main():
    # Set your Hugging Face token here
    HF_TOKEN = None  # Or set to your token string
    
    if HF_TOKEN is None:
        print("WARNING: No Hugging Face token provided.")
        print("You may need to set HF_TOKEN to access Gemma 2.")
        print("Get a token at: https://huggingface.co/settings/tokens")
    
    # Load model
    model, tokenizer = load_gemma2(token=HF_TOKEN)
    
    # Process all clues
    all_results = {}
    
    for clue_num, clue_data in CLUES.items():
        results = analyze_clue(model, tokenizer, clue_num, clue_data)
        all_results[clue_num] = results
    
    # Save results
    print("\n" + "="*80)
    print("SAVING RESULTS...")
    print("="*80)
    
    np.savez('crossword_logits.npz', **{
        f"{clue_num}_{word}": logits 
        for clue_num, candidates in all_results.items()
        for word, logits in candidates.items()
    })
    
    print("\nResults saved to 'crossword_logits.npz'")
    print("\nTo load results:")
    print("  data = np.load('crossword_logits.npz')")
    print("  print(data.files)  # List all saved arrays")
    print("  logits = data['1a_SOMEWORD']  # Access specific word's logits")

if __name__ == "__main__":
    main()

You may need to set HF_TOKEN to access Gemma 2.
Get a token at: https://huggingface.co/settings/tokens
Loading google/gemma-2-9b-it...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded successfully!

CLUE 1a: Acquisitive chap, as we see it (8)

Found 4 candidates:

Candidate: AMASSING


TypeError: Got unsupported ScalarType BFloat16